# 02 VLE Fairness Audit Using OULAD

This notebook audits VLE engagement analytics using the processed OULAD feature table.



In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

REPO_ROOT = Path.cwd()
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
input_path = PROCESSED_DIR / "oulad_student_features.csv"

print("Input path:", input_path)
print("Available:", input_path.exists())

In [ ]:
if input_path.exists():
    df = pd.read_csv(input_path)
    print("Loaded:", df.shape)
else:
    df = pd.DataFrame()
    print("Missing file. Run 01_oulad_data_preparation.ipynb first.")

In [ ]:
if not df.empty:
    df.head()

## Representation Analysis

This checks whether gender groups and other student groups have sufficient representation for fairness analysis.

In [ ]:
if not df.empty:
    representation_summary = (
        df.groupby("gender", dropna=False)
        .size()
        .reset_index(name="student_count")
    )
    representation_summary["percentage"] = (
        representation_summary["student_count"] / representation_summary["student_count"].sum() * 100
    ).round(2)
    representation_summary["sample_size_warning"] = representation_summary["student_count"] < 100

    representation_summary.to_csv(
        PROCESSED_DIR / "oulad_representation_summary.csv",
        index=False
    )

    representation_summary

## Fairness Audit

This demonstration uses two outcome-style indicators:

- `is_unsuccessful_outcome`
- `low_engagement_flag`

These are not final judgements about students. They are used to demonstrate how learning analytics indicators can be audited across gender groups.

In [ ]:
if not df.empty:
    fairness_summary = (
        df.groupby("gender", dropna=False)
        .agg(
            student_count=("id_student", "count"),
            avg_total_vle_clicks=("total_vle_clicks", "mean"),
            avg_assessment_score=("assessment_score_avg", "mean"),
            unsuccessful_outcome_rate=("is_unsuccessful_outcome", "mean"),
            low_engagement_flag_rate=("low_engagement_flag", "mean"),
        )
        .reset_index()
    )

    fairness_summary["avg_total_vle_clicks"] = fairness_summary["avg_total_vle_clicks"].round(2)
    fairness_summary["avg_assessment_score"] = fairness_summary["avg_assessment_score"].round(2)
    fairness_summary["unsuccessful_outcome_rate"] = (fairness_summary["unsuccessful_outcome_rate"] * 100).round(2)
    fairness_summary["low_engagement_flag_rate"] = (fairness_summary["low_engagement_flag_rate"] * 100).round(2)

    fairness_summary.to_csv(
        PROCESSED_DIR / "oulad_vle_fairness_summary.csv",
        index=False
    )

    fairness_summary

In [ ]:
if not df.empty:
    plot_df = fairness_summary.sort_values("low_engagement_flag_rate", ascending=False)

    plt.figure(figsize=(8, 5))
    plt.bar(plot_df["gender"].astype(str), plot_df["low_engagement_flag_rate"])
    plt.title("OULAD VLE Audit: Low Engagement Flag Rate by Gender")
    plt.xlabel("Gender")
    plt.ylabel("Low engagement flag rate (%)")
    plt.tight_layout()
    plt.show()

## Proxy Association Checks

This section uses Cramer's V to check whether categorical variables are associated with gender.

Higher association does not automatically mean unlawful bias. It means the feature needs governance review because it may act as a proxy.

In [ ]:
def cramers_v(x: pd.Series, y: pd.Series) -> float:
    """Calculate Cramer's V association between two categorical variables."""
    table = pd.crosstab(x.fillna("Missing"), y.fillna("Missing"))
    if table.shape[0] < 2 or table.shape[1] < 2:
        return 0.0

    chi2 = chi2_contingency(table)[0]
    n = table.sum().sum()
    r, k = table.shape

    denominator = n * (min(k - 1, r - 1))
    if denominator == 0:
        return 0.0

    return float(np.sqrt(chi2 / denominator))


if not df.empty:
    candidate_proxy_columns = [
        "region",
        "highest_education",
        "imd_band",
        "age_band",
        "num_of_prev_attempts",
        "studied_credits",
        "disability",
        "code_module",
    ]

    available_proxy_columns = [col for col in candidate_proxy_columns if col in df.columns]

    proxy_rows = []
    for col in available_proxy_columns:
        score = cramers_v(df[col], df["gender"])
        if score < 0.10:
            risk_band = "Very low"
        elif score < 0.20:
            risk_band = "Low"
        elif score < 0.30:
            risk_band = "Moderate"
        else:
            risk_band = "High"

        proxy_rows.append({
            "variable": col,
            "association_with_gender_cramers_v": round(score, 3),
            "proxy_risk_band": risk_band,
        })

    proxy_summary = pd.DataFrame(proxy_rows).sort_values(
        "association_with_gender_cramers_v",
        ascending=False,
    )

    proxy_summary.to_csv(
        PROCESSED_DIR / "oulad_proxy_association_summary.csv",
        index=False,
    )

    proxy_summary

## Interpretation

> This audit identifies where VLE analytics indicators and student variables require fairness review before being used in intervention, flagging, or support decisions.